In [1]:
import pandas as pd
import numpy as np

In [5]:
# 간단한 영화 데이터 생성
movie_data = {
    'movie_id': [101, 102, 103, 104, 105],
    'title': ['Movie A', 'Movie B', 'Movie C', 'Movie D', 'Movie E'],
    'genre': ['Action', 'Comedy', 'Action', 'Comedy', 'Drama']
}

user_ratings = {
    'user_id': [1, 1, 1, 2, 2],
    'movie_id': [101, 102, 103, 104, 105],
    'rating': [5, 3, 4, 4, 5]
}

movie_df = pd.DataFrame(movie_data)
user_ratings_df = pd.DataFrame(user_ratings)

# 장르를 one-hot encoding하여 콘텐츠 기반 행렬 생성
movie_df['genre_encoded'] = pd.get_dummies(movie_df['genre']).values.tolist()

# 특정 사용자가 평가한 영화의 장르 벡터를 가져옴
target_user_ratings = user_ratings_df[user_ratings_df['user_id'] == 1]
rated_movies = movie_df[movie_df['movie_id'].isin(target_user_ratings['movie_id'])]
user_genre_profile = np.mean(np.array(rated_movies['genre_encoded'].tolist()), axis=0)
print(user_genre_profile)

# 나머지 영화와의 유사도 계산
movie_df['similarity'] = movie_df['genre_encoded'].apply(lambda x: np.dot(user_genre_profile, x))

# 추천할 영화 선택 (사용자가 보지 않은 영화)
recommendations = movie_df[~movie_df['movie_id'].isin(target_user_ratings['movie_id'])].sort_values(by='similarity', ascending=False)
print(f"Recommendations for user 1 based on content: \n{recommendations[['title', 'similarity']]}")

movie_df

[0.66666667 0.33333333 0.        ]
Recommendations for user 1 based on content: 
     title  similarity
3  Movie D    0.333333
4  Movie E    0.000000


,movie_id,title,genre,genre_encoded,similarity
0,101,Movie A,Action,"[True, False, False]",0.666667
1,102,Movie B,Comedy,"[False, True, False]",0.333333
2,103,Movie C,Action,"[True, False, False]",0.666667
3,104,Movie D,Comedy,"[False, True, False]",0.333333
4,105,Movie E,Drama,"[False, False, True]",0.000000


콘텐츠 기반 추천 시스템
- 봤거나 좋아하는 아이템의 특징을 바탕으로 유사 아이템 추천
- 각 영화 장르 벡터와 사용자 취향 프로필 유사도 계산
- 사용자가 보지 않은 영화 중 유사도 높은 것 추천

콘텐츠 기반 추천 시스템 - 아이템 중심
- 내가 좋아하는 아이템의 속성과 비슷한 속성 가진 아이템 추천
- 사용자가 안 보거나 좋아한 적 없는 속성 가진 아이템 추천 어려움 (콜드 스타트 문제)

협업 필터링 추천 시스템 = 사용자 중심
- 나랑 비슷한 사람이 좋아하는 아이템 추천
- 데이터가 없으면 실용성 없을 수 있다

In [6]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 간단한 사용자-아이템 평점 데이터 생성
data = {
    'user_id': [1, 1, 1, 2, 2, 3, 3, 4, 4, 4],
    'item_id': [101, 102, 103, 101, 104, 101, 102, 103, 104, 105],
    'rating': [5, 3, 4, 4, 5, 3, 4, 2, 4, 5]
}

df = pd.DataFrame(data)
user_item_matrix = df.pivot_table(index='user_id', columns='item_id', values='rating').fillna(0)

# 사용자 간 유사도 계산 (코사인 유사도)
user_similarity = cosine_similarity(user_item_matrix)
user_similarity_df = pd.DataFrame(user_similarity, index=user_item_matrix.index, columns=user_item_matrix.index)

# 특정 사용자의 추천 아이템 (예: user_id=1)
target_user = 1
similar_users = user_similarity_df[target_user].sort_values(ascending=False).index[1:]

# 유사한 사용자의 아이템 중에서, target_user가 평가하지 않은 아이템 추천
items_rated_by_target = user_item_matrix.loc[target_user, user_item_matrix.loc[target_user] > 0].index
recommendations = []
for user in similar_users:
    items = user_item_matrix.loc[user, user_item_matrix.loc[user] > 0].index
    new_recommendations = [item for item in items if item not in items_rated_by_target]
    recommendations.extend(new_recommendations)
    if len(recommendations) >= 2:
        break

print(f"Recommendations for user {target_user}: {recommendations}")

Recommendations for user 1: [104, 104, 105]


In [7]:
# 협업 필터링을 통한 추천 결과 (간단한 예시)
collaborative_recommendations = [101, 104]

# 콘텐츠 기반 필터링을 통한 추천 결과
content_recommendations = [105, 102]

# 하이브리드 추천 (가중치 결합)
final_recommendations = list(set(collaborative_recommendations + content_recommendations))
print(f"Hybrid Recommendations: {final_recommendations}")

Hybrid Recommendations: [104, 105, 101, 102]


협업 필터링
- 사용자의 평점 패턴을 비교하고, 비슷한 사용자가 좋아한 아이템 추천하는 방식
- 사용자-아이템의 평점 행렬 만든 후, 코사인 유사도로 비슷한 사용자 찾는다
- 이후 target_user가 아직 평가(관람, 구매)하지 않은 아이템 추천

In [8]:
# 협업 필터링을 통한 추천 결과 (간단한 예시)
collaborative_recommendations = [101, 104]

# 콘텐츠 기반 필터링을 통한 추천 결과
content_recommendations = [105, 102]

# 하이브리드 추천 (가중치 결합)
final_recommendations = list(set(collaborative_recommendations + content_recommendations))
print(f"Hybrid Recommendations: {final_recommendations}")

Hybrid Recommendations: [104, 105, 101, 102]
